In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.ndimage import gaussian_filter1d
import ipywidgets as widgets
from IPython.display import display

# ===================================================================
# 1. Parameters (from your slip_adjoint_springslider_adapttime notebook)
# ===================================================================
M_true = {
    'f0': 0.6,
    'V0': 1e-6,
    'a': 0.010,
    'b': 0.015,
    'dc': 1e-4,
    'N': 50.0,
    'eta': 2.7 * 3.5 / 2.0,
    'V_bg': 1e-9
}
# Fixed k based on true a=0.010
k_crit_true = M_true['N'] * (M_true['b'] - M_true['a']) / M_true['dc']
M_true['k'] = 0.9 * k_crit_true


# ===================================================================
# 2. Physics & Forward Solver
# ===================================================================
def tau_fn(V, psi, M):
    xi = V / (2.0 * M['V0']) * np.exp(psi / M['a'])
    return M['N'] * M['a'] * np.arcsinh(xi)

def fss_fn(V, M):
    return M['f0'] + (M['a'] - M['b']) * np.log(V / M['V0'])

def solve_V_algebraic(u, psi, M, tau_L):
    rhs = tau_L - M['k'] * u
    def res(V):
        return tau_fn(V, psi, M) + M['eta'] * V - rhs
    
    # Robust root-finding bounds
    Vmin, Vmax = 1e-20, 10.0
    if res(Vmin) * res(Vmax) > 0:
        Vmin = 1e-25
    return brentq(res, Vmin, Vmax)

def setup_initial_conditions(M):
    fss_bg = fss_fn(M['V_bg'], M)
    psi_ss = M['a'] * np.log(2.0 * M['V0'] / M['V_bg'] * np.sinh(fss_bg / M['a']))
    V_init = 1.0e-12
    M['tau0'] = tau_fn(V_init, psi_ss, M) + M['eta'] * V_init
    return 0.0, psi_ss, V_init

def forward_solve_adaptive(M, T, u0, psi0, V_init):
    """Adaptive RK3 solver matching your adapt_fwd_solve.py"""
    tau_L_fn = lambda t: M['tau0'] + M['k'] * M['V_bg'] * t
    t, u, psi, dt = 0.0, u0, psi0, 1.0
    tol, dtmax, safety = 1e-8, 1e5, 0.8
    t_arr, u_arr = [t], [u]
    
    def _rhs(u_v, psi_v, t_v):
        V = solve_V_algebraic(u_v, psi_v, M, tau_L_fn(t_v))
        f = tau_fn(V, psi_v, M) / M['N']
        G = -V / M['dc'] * (f - fss_fn(V, M))
        return V, G
        
    while t < T:
        if t + dt > T: dt = T - t
        V1, G1 = _rhs(u, psi, t)
        V2, G2 = _rhs(u + 0.5*dt*V1, psi + 0.5*dt*G1, t + 0.5*dt)
        V3, G3 = _rhs(u + dt*(-V1 + 2.0*V2), psi + dt*(-G1 + 2.0*G2), t + dt)
        
        u2 = u + dt/2.0*(V1 + V3)
        psi2 = psi + dt/2.0*(G1 + G3)
        u3 = u + dt/6.0*(V1 + 4.0*V2 + V3)
        psi3 = psi + dt/6.0*(G1 + 4.0*G2 + G3)
        
        er = np.sqrt((u2 - u3)**2 + (psi2 - psi3)**2)
        if er < tol:
            t += dt; u = u3; psi = psi3
            t_arr.append(t); u_arr.append(u)
        dt = min(safety * dt * (tol / er)**(1./3.), dtmax) if er > 0 else dtmax
        
    return np.array(t_arr), np.array(u_arr)


# ===================================================================
# 3. Precompute data (Observed + grid of Model parameters)
# ===================================================================
T = 1e7
print("1/2: Precomputing observed (true) earthquake cycle (a=0.010)...")
u0, psi0, V_init = setup_initial_conditions(M_true)
t_obs, u_obs = forward_solve_adaptive(M_true, T, u0, psi0, V_init)

# Interpolate to a common grid so S_matrix math is extremely fast for interactivity
t_grid = np.linspace(0, T, 1000)
u_obs_grid = np.interp(t_grid, t_obs, u_obs)

a_vals = np.linspace(0.008, 0.02, 40)
u_mods_grid = []

print("2/2: Precomputing model variations across 'a' values.")
for a_val in a_vals:
    M_mod = M_true.copy()
    M_mod['a'] = a_val
    u0_m, psi0_m, V_init_m = setup_initial_conditions(M_mod)
    t_m, u_m = forward_solve_adaptive(M_mod, T, u0_m, psi0_m, V_init_m)
    u_mods_grid.append(np.interp(t_grid, t_m, u_m))
print("Precomputation complete! Launching interactive widget...")


# ===================================================================
# 4. Interactive Widget & Landscape calculation
# ===================================================================
def apply_smoothing_uniform(u, t_grid, sigma_time):
    # If sigma is tiny relative to T, return raw steps
    if sigma_time <= 1e4:
        return u
    dt = t_grid[1] - t_grid[0]
    sigma_pixels = sigma_time / dt
    return gaussian_filter1d(u, sigma_pixels, mode='nearest')

def plot_interactive(a_idx, sigma_time):
    S_u_obs = apply_smoothing_uniform(u_obs_grid, t_grid, sigma_time)
    
    J_array = []
    # Calculate the entire objective landscape given current smoothing
    for u_mod in u_mods_grid:
        S_u_mod = apply_smoothing_uniform(u_mod, t_grid, sigma_time)
        J = 0.5 * np.trapz((S_u_mod - S_u_obs)**2, t_grid)
        J_array.append(J)
        
    current_u_mod = u_mods_grid[a_idx]
    S_current_u_mod = apply_smoothing_uniform(current_u_mod, t_grid, sigma_time)
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 9))
    
    # ---- Top panel: Time series ----
    ax1.plot(t_grid, S_u_obs, 'k--', lw=2, label='Observed Data (a=0.0100)')
    ax1.plot(t_grid, S_current_u_mod, 'r-', lw=2, label=f'Model Data (a={a_vals[a_idx]:.4f})')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Cumulative Slip $u(t)$')
    ax1.legend(loc='upper left')
    ax1.set_title(f'Slip Histories ($\sigma$ = {sigma_time:.1e} s)')
    ax1.grid(True, alpha=0.3)
    
    # ---- Bottom panel: Objective landscape ----
    ax2.plot(a_vals, J_array, 'b-', lw=2, label='L2 Objective $J$')
    ax2.plot(a_vals[a_idx], J_array[a_idx], 'ro', markersize=10, label='Current Model State')
    ax2.axvline(0.010, color='k', linestyle='--', label='Global Minimum ($a_{true}$)')
    ax2.set_xlabel('Friction Parameter $a$')
    ax2.set_ylabel('Objective $J$')
    
    # Scale toggles to log if variation is massive (typical with raw L2)
    if np.max(J_array) / (np.min(J_array) + 1e-12) > 50:
        ax2.set_yscale('log')
        
    ax2.legend(loc='upper right')
    ax2.set_title("Error Landscape Geometry (Observe the 'Flat Plains' vs 'Parabola')")
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Setup UI Controls
a_slider = widgets.IntSlider(min=0, max=len(a_vals)-1, value=15, description='Model $a$ idx:')
# Using a FloatLogSlider so we can sweep sigma from 10,000s up to 10,000,000s exponentially
sigma_slider = widgets.FloatLogSlider(min=4, max=7.5, value=1e4, step=0.05, description='Smoothing $\sigma$ (s):', readout_format='.1e')

out = widgets.interactive_output(plot_interactive, {'a_idx': a_slider, 'sigma_time': sigma_slider})
display(widgets.HBox([a_slider, sigma_slider]))
display(out)

1/2: Precomputing observed (true) earthquake cycle (a=0.010)...
2/2: Precomputing model variations across 'a' values.
Precomputation complete! Launching interactive widget...


Output()